# CNN Distribution Shift — Full Training on GrainSet Wheat

This notebook:
1. Clones your GitHub repo
2. Downloads the GrainSet wheat dataset from Figshare
3. Runs the data pipeline
4. Trains ResNet-50 for 50 epochs on a GPU
5. Downloads the checkpoint to your Google Drive

**Before running:** Make sure you selected a GPU runtime:
Runtime → Change runtime type → T4 GPU

## Step 1 — Check GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 2 — Clone your GitHub repo

**Edit the REPO_URL below** to point to your GitHub repository.

In [ ]:
# --- EDIT THIS ---
REPO_URL = 'https://github.com/YOUR_USERNAME/cnn-distribution-shift.git'
# -----------------

import os
!git clone {REPO_URL}

# Change into the repo directory
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')
os.chdir(REPO_NAME)
!ls

## Step 3 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Done')

## Step 4 — Download GrainSet wheat dataset

~6 GB. Takes about 3-5 minutes on Colab's fast connection.

In [ ]:
!mkdir -p data/wheat

# Download wheat.zip from Figshare
!wget -q --show-progress -O data/wheat.zip \
    'https://figshare.com/ndownloader/files/40902838'

print('Download complete. Unzipping...')
!unzip -q data/wheat.zip -d data/wheat/
!rm data/wheat.zip

print('Done. Contents:')
!find data/wheat -type f | head -10
!find data/wheat -name '*.xml'

## Step 5 — Inspect the downloaded structure

Run this to confirm the paths before preparing data.

In [ ]:
import os

# Find the XML file
for root, dirs, files in os.walk('data/wheat'):
    for f in files:
        if f.endswith('.xml'):
            print('XML found:', os.path.join(root, f))

# Count images
png_count = sum(1 for _, _, files in os.walk('data/wheat')
                for f in files if f.endswith('.png') and 'mask' not in _)
print(f'PNG images (excl. masks): {png_count:,}')

# Show directory structure
!find data/wheat -type d | head -20

## Step 6 — Prepare data pipeline

**Edit XML_PATH and ROOT_PATH** based on the output of Step 5.

In [ ]:
# --- EDIT THESE if Step 5 shows different paths ---
XML_PATH  = 'data/wheat/wheat.xml'   # path to the XML annotation file
ROOT_PATH = 'data/wheat/'            # directory containing train/ and test/
DATALIST  = 'runs/datalist/wheat'
# --------------------------------------------------

!python src/data/prepare_data.py \
    --xml  {XML_PATH} \
    --root {ROOT_PATH} \
    --out  {DATALIST}

## Step 7 — Train ResNet-50 (50 epochs, GPU)

Expected time: ~2-4 hours on a T4 GPU.

The best checkpoint is saved to `runs/checkpoints/wheat_resnet50/best.pth`.

In [ ]:
CHECKPOINT_DIR = 'runs/checkpoints/wheat_resnet50'

!python src/models/train.py \
    --datalist    {DATALIST} \
    --out         {CHECKPOINT_DIR} \
    --epochs      50 \
    --batch-size  128 \
    --device      cuda \
    --num-workers 2

## Step 8 — Save checkpoint to Google Drive

Colab sessions reset after a few hours, so save the checkpoint
to your Google Drive to keep it permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Create destination folder in Drive
dest = '/content/drive/MyDrive/cnn-distribution-shift/checkpoints'
os.makedirs(dest, exist_ok=True)

# Copy best checkpoint
shutil.copy('runs/checkpoints/wheat_resnet50/best.pth', dest)
shutil.copy('runs/checkpoints/wheat_resnet50/train_log.csv', dest)

# Also save the datalist files
shutil.copytree('runs/datalist/wheat',
                '/content/drive/MyDrive/cnn-distribution-shift/datalist',
                dirs_exist_ok=True)

print('Saved to Google Drive:')
!ls /content/drive/MyDrive/cnn-distribution-shift/checkpoints/

## Step 9 — Plot training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('runs/checkpoints/wheat_resnet50/train_log.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(log['epoch'], log['train_loss'], label='Train')
ax1.plot(log['epoch'], log['val_loss'],   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(log['epoch'], log['train_f1'], label='Train')
ax2.plot(log['epoch'], log['val_f1'],   label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1')
ax2.set_title('F1 Score'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('runs/checkpoints/wheat_resnet50/training_curves.pdf',
            bbox_inches='tight')
plt.show()

best_row = log.loc[log['val_f1'].idxmax()]
print(f"Best val F1 = {best_row['val_f1']:.4f} at epoch {int(best_row['epoch'])}")

## Step 10 — Run baseline curve experiment

In [ ]:
!python src/experiments/baseline_curve.py \
    --datalist   {DATALIST} \
    --checkpoint runs/checkpoints/wheat_resnet50/best.pth \
    --out        runs/results/wheat \
    --num-workers 2

# Save results to Drive
shutil.copytree('runs/results/wheat',
                '/content/drive/MyDrive/cnn-distribution-shift/results',
                dirs_exist_ok=True)
print('Results saved to Drive')